# Pic2Model on Colab GPU (Hunyuan3D-2.0 + PBR texture)
Runs the FastAPI backend on a Colab T4 GPU and exposes it via a Cloudflare tunnel.
Point your **local** Next.js frontend at the printed `BACKEND_URL`.

**Runtime → Change runtime type → GPU (T4)** before running. Run cells top to bottom.

Using 2.0 (not 2.1): 2.1 OOMs Colab's free ~13GB RAM at load. 2.0 fits, runs fast on the 15GB GPU, and its texture pipeline compiles cleanly on Linux.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone the backend repo
REPO_URL = 'https://github.com/PeerawatProject14/Pic2Model.git'
%cd /content
![ -d Pic2Model ] && rm -rf Pic2Model
!git clone $REPO_URL Pic2Model
%cd /content/Pic2Model

In [ ]:
# 3. Clone Hunyuan3D-2.0 into backend/vendor
!mkdir -p backend/vendor
![ -d backend/vendor/Hunyuan3D-2 ] || git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git backend/vendor/Hunyuan3D-2

In [ ]:
# 4. Install deps  (transformers pinned to 4.46.3 to match the 2.0 checkpoint)
!pip -q install fastapi 'uvicorn[standard]' python-multipart pydantic pydantic-settings pillow 'numpy<2.1' trimesh pygltflib shapely rembg onnxruntime
!pip -q install 'transformers==4.46.3' diffusers accelerate einops opencv-python scikit-image pymeshlab xatlas omegaconf pybind11 ninja
# compile texture extensions (NON-editable so the running kernel can import them right away)
%cd /content/Pic2Model/backend/vendor/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!pip install . --no-build-isolation 2>&1 | tail -4
%cd /content/Pic2Model/backend/vendor/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!pip install . --no-build-isolation 2>&1 | tail -4
%cd /content/Pic2Model
import importlib; importlib.invalidate_caches()
import custom_rasterizer; print('custom_rasterizer OK ->', custom_rasterizer.__file__)

In [ ]:
# 5. Configure backend: Hunyuan3D-2.0 + texture ON
env = '''PIC2MODEL_GENERATOR_BACKEND=hunyuan3d
PIC2MODEL_BG_BACKEND=rembg
PIC2MODEL_SEGMENTER_BACKEND=manual
PIC2MODEL_DEVICE=cuda
PIC2MODEL_TEXTURE=true
PIC2MODEL_DEFAULT_TARGET_FACES=150000
PIC2MODEL_CORS_ORIGINS=["*"]
'''
open('backend/.env','w').write(env)
print(env)

In [ ]:
# 6. (optional) smoke-test the model loads on GPU before serving
import sys; sys.path.insert(0, '/content/Pic2Model/backend')
import os
from PIL import Image
from app.pipeline.hunyuan3d import Hunyuan3DGenerator
demo = 'backend/vendor/Hunyuan3D-2/assets/example_images/004.png'
img = Image.open(demo).convert('RGBA') if os.path.exists(demo) else Image.new('RGBA',(512,512),(180,140,90,255))
g = Hunyuan3DGenerator(device_hint='cuda', octree_resolution=256, num_inference_steps=30, texture=False)
m = g.generate(img); print('SHAPE OK:', len(m.vertices), 'verts'); g.unload()

In [ ]:
# 7. Start backend + Cloudflare tunnel (uvicorn log -> /content/backend.log)
import subprocess, time, re, os
for d in ('inputs','models','thumbnails'):
    os.makedirs(f'/content/Pic2Model/storage/{d}', exist_ok=True)
logf = open('/content/backend.log','w')
be = subprocess.Popen(['python','-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'],
                      cwd='/content/Pic2Model/backend', stdout=logf, stderr=subprocess.STDOUT)
time.sleep(8)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
tun = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for line in tun.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n==============================================')
print('  BACKEND_URL =', url)
print('==============================================')

In [ ]:
# 8. Watch backend log live (run to see generation progress / errors)
!tail -n 80 -f /content/backend.log